In [1]:
from __future__ import annotations
from geopy.geocoders import Nominatim
from dataclasses import dataclass
import xarray as xr
import pandas as pd
import numpy as np

@dataclass
class Location:
    name: str
    latitude: float
    longitude: float

_geolocator = None

def geocode_city(city: str, user_agent: str = "pogoda-app") -> Location:
    """Geocode a city name into coordinates using Nominatim.

    Parameters
    ----------
    city: str
        City name (can include country, e.g., "Szczecin, Poland").
    user_agent: str
        Required by Nominatim usage policy.

    Returns
    -------
    Location

    Raises
    ------
    ValueError: if no result is found.
    """
    global _geolocator
    if _geolocator is None:
        _geolocator = Nominatim(user_agent=user_agent, timeout=10)
    result = _geolocator.geocode(city)
    if not result:
        raise ValueError(f"City not found: {city}")
    return Location(name=result.address, latitude=result.latitude, longitude=result.longitude)


In [11]:
temp = xr.open_dataset("tg_ens_mean_0.1deg_reg_v31.0e.nc") #tg_ens_mean_0.1deg_reg_2011-2024_v31.0e.nc
rain = xr.open_dataset("rr_ens_mean_0.1deg_reg_v31.0e.nc")

In [3]:
def get_data(da: xr.DataArray, city, agg: str = "mean", box_radius_deg: float | None = None):
    """
    da  : xarray.DataArray (e.g., ds['tg'] or ds['rr'])
    city: object with .latitude and .longitude (floats)
    agg : 'mean' (e.g., TG) or 'sum' (e.g., RR)
    box_radius_deg: half-width of the lat/lon box in degrees.
    """
    # --- pick a safe radius that guarantees at least one cell ---
    if box_radius_deg is None:
        lat_res = float(abs(da.latitude.values[1] - da.latitude.values[0]))
        lon_res = float(abs(da.longitude.values[1] - da.longitude.values[0]))
        box_radius_deg = 0.5 * max(lat_res, lon_res)

    # --- select box ---
    box = da.sel(
        latitude=slice(city.latitude - box_radius_deg, city.latitude + box_radius_deg),
        longitude=slice(city.longitude - box_radius_deg, city.longitude + box_radius_deg),
    )

    # fallback to nearest cell if empty
    if box.sizes.get("latitude", 0) == 0 or box.sizes.get("longitude", 0) == 0:
        box = da.sel(latitude=city.latitude, longitude=city.longitude, method="nearest")

    # --- monthly aggregation ---
    if agg == "mean":
        monthly = box.resample(time="1MS").mean(dim=("time", "latitude", "longitude"))
    elif agg == "sum":
        monthly = box.resample(time="1MS").sum(dim=("time", "latitude", "longitude"))
    else:
        raise ValueError("agg must be 'mean' or 'sum'")

    # --- counts ---
    da_nonmissing = box.resample(time="1MS").count(dim=("time", "latitude", "longitude"))

    n_lat = box.sizes.get("latitude", 1)
    n_lon = box.sizes.get("longitude", 1)
    n_cells = n_lat * n_lon

    days_per_month = box["time"].resample(time="1MS").count()
    total_vals = days_per_month * n_cells
    da_missing = total_vals - da_nonmissing

    # --- assemble DataFrame ---
    df = pd.DataFrame({
        "time": pd.to_datetime(monthly["time"].values),
        "value": monthly.values,
        "count_nonmissing": da_nonmissing.values.astype(np.int64),
        "count_missing": da_missing.values.astype(np.int64),
    })
    df["year"] = df["time"].dt.year
    df["month"] = df["time"].dt.month
    return df[["year", "month", "value", "count_nonmissing", "count_missing"]]

In [4]:
from __future__ import annotations
from typing import TypedDict
import pandas as pd
import numpy as np


class KoppenClassificationError(Exception):
    pass


class KoppenDetails(TypedDict, total=False):
    annual_mean_temp: float
    annual_precip: float
    warmest: float
    coldest: float
    months_ge_10: int
    precip_summer: float
    precip_winter: float
    summer_share: float
    winter_share: float
    dryness_threshold_R: float
    group: str
    second: str
    third: str
    sub1: str
    sub2: str
    subtype: str


def _climatology_from_year_month(
    df: pd.DataFrame,
    *,
    temp_col: str | None,
    precip_col: str | None,
) -> pd.DataFrame:
    """
    Build a 12-row monthly climatology from year/month columns.

    - Expects df to contain: 'year', 'month' and one or both of temp_col/precip_col.
    - If there are multiple rows per (year, month), we:
        temp  : average within the month
        precip: sum within the month
      Then we average across years to get a typical month.
    """
    need_cols = {"year", "month"}
    if not need_cols.issubset(df.columns):
        raise KoppenClassificationError("DataFrame must have 'year' and 'month' columns.")

    agg_map: dict[str, str] = {}
    if temp_col and temp_col in df.columns:
        agg_map[temp_col] = "mean"      # average temp within month
    if precip_col and precip_col in df.columns:
        agg_map[precip_col] = "sum"     # sum precip within month
    if not agg_map:
        raise KoppenClassificationError("No temperature or precipitation column found.")

    # 1) collapse to one row per (year, month)
    per_ym = (
        df.groupby(["year", "month"], as_index=False)
          .agg(agg_map)
    )

    # 2) average across years -> 12 rows (one per calendar month)
    climat = (
        per_ym.groupby("month", as_index=False)
              .agg({k: "mean" for k in agg_map})
              .sort_values("month")
              .reset_index(drop=True)
    )
    return climat


def classify_koppen_from_tables(
    *,
    latitude: float,
    temp_df: pd.DataFrame | None = None,      # expects year, month, temp
    precip_df: pd.DataFrame | None = None,    # expects year, month, precip
    temp_col: str = "temp",
    precip_col: str = "rain",
) -> tuple[str, KoppenDetails]:
    """
    Köppen classification using year/month tables.

    You can pass:
      • a single DataFrame via temp_df that already has both columns (temp_col & precip_col), OR
      • two separate DataFrames: temp_df (year, month, temp) and precip_df (year, month, precip).

    The function builds a 12-month climatology:
      temp: mean over years for each month
      precip: mean monthly total over years
    """
    if temp_df is None and precip_df is None:
        raise KoppenClassificationError("Provide at least one DataFrame.")

    if temp_df is not None and precip_df is None and precip_col in temp_df.columns:
        # single table with both columns
        climat = _climatology_from_year_month(temp_df, temp_col=temp_col, precip_col=precip_col)
    else:
        # handle separate tables (or temp-only table merged with precip-only table)
        parts = []
        if temp_df is not None and temp_col in temp_df.columns:
            c_temp = _climatology_from_year_month(temp_df, temp_col=temp_col, precip_col=None)
            parts.append(c_temp)
        if precip_df is not None and precip_col in precip_df.columns:
            c_prec = _climatology_from_year_month(precip_df, temp_col=None, precip_col=precip_col)
            parts.append(c_prec)
        if not parts:
            raise KoppenClassificationError("Could not find required columns in provided tables.")
        # merge by month
        climat = parts[0]
        for p in parts[1:]:
            climat = climat.merge(p, on="month", how="inner")

    # Require full 12 months
    if len(climat) != 12 or set(climat["month"]) != set(range(1, 13)):
        raise KoppenClassificationError("After aggregation, expected 12 months (1..12).")

    # Extract arrays
    if temp_col not in climat.columns or precip_col not in climat.columns:
        missing = [c for c in (temp_col, precip_col) if c not in climat.columns]
        raise KoppenClassificationError(f"Missing required monthly columns: {missing}")

    temps_c = climat[temp_col].to_numpy(dtype=float)
    precip_mm = climat[precip_col].to_numpy(dtype=float)

    if np.any(np.isnan(temps_c)) or np.any(np.isnan(precip_mm)):
        raise KoppenClassificationError("Climatology contains NaNs; clean your inputs first.")

    # ---- Köppen logic (same as before) ----
    annual_mean_temp = float(np.mean(temps_c))
    annual_precip = float(np.sum(precip_mm))
    warmest = float(np.max(temps_c))
    coldest = float(np.min(temps_c))
    months_ge_10 = int(np.sum(temps_c >= 10.0))

    if latitude >= 0:
        summer_idx = np.array([4,5,6,7,8,9]) - 1  # Apr–Sep -> months 4..9 -> indices 3..8
        winter_idx = np.array([10,11,12,1,2,3]) - 1
    else:
        summer_idx = np.array([10,11,12,1,2,3]) - 1
        winter_idx = np.array([4,5,6,7,8,9]) - 1

    precip_summer = float(np.sum(precip_mm[summer_idx]))
    precip_winter = float(np.sum(precip_mm[winter_idx]))

    if annual_precip > 0:
        summer_share = precip_summer / annual_precip
        winter_share = precip_winter / annual_precip
    else:
        summer_share = 0.0
        winter_share = 0.0

    # Dryness threshold R
    if summer_share >= 0.70:
        R = 2 * annual_mean_temp + 28
    elif winter_share >= 0.70:
        R = 2 * annual_mean_temp
    else:
        R = 2 * annual_mean_temp + 14

    details: KoppenDetails = {
        "annual_mean_temp": annual_mean_temp,
        "annual_precip": annual_precip,
        "warmest": warmest,
        "coldest": coldest,
        "months_ge_10": months_ge_10,
        "precip_summer": precip_summer,
        "precip_winter": precip_winter,
        "summer_share": summer_share,
        "winter_share": winter_share,
        "dryness_threshold_R": R,
    }

    # Main group (order matters)
    if annual_precip < R:
        main = "B"
    elif coldest >= 18.0:
        main = "A"
    elif warmest < 10.0:
        main = "E"
    elif coldest > -3.0:
        main = "C"
    else:
        main = "D"

    # Polar
    if main == "E":
        subtype = "T" if warmest > 0.0 else "F"
        details.update(group=main, subtype=subtype)
        return main + subtype, details

    # Arid
    if main == "B":
        sub1 = "W" if annual_precip < 0.5 * R else "S"
        sub2 = "h" if annual_mean_temp >= 18.0 else "k"
        details.update(group=main, sub1=sub1, sub2=sub2)
        return main + sub1 + sub2, details

    # Seasonality second letter (A, C, D)
    summer_months = precip_mm[summer_idx]
    winter_months = precip_mm[winter_idx]
    driest_summer = float(np.min(summer_months)) if summer_months.size else 0.0
    driest_winter = float(np.min(winter_months)) if winter_months.size else 0.0
    wettest_winter = float(np.max(winter_months)) if winter_months.size else 0.0
    wettest_summer = float(np.max(summer_months)) if summer_months.size else 0.0

    if (driest_summer < 40.0) and (wettest_winter > 0.0) and (driest_summer < (wettest_winter / 3.0)):
        second = "s"
    elif (wettest_summer > 0.0) and (driest_winter < (wettest_summer / 10.0)):
        second = "w"
    else:
        second = "f"

    if main == "A":
        if np.all(precip_mm >= 60.0):
            second = "f"
        else:
            if np.any(precip_mm[winter_idx] < 60.0) and (precip_winter < precip_summer):
                second = "w"
            else:
                second = "m"
        details.update(group=main, second=second)
        return main + second, details

    # Third letter for C/D
    if main in ("C", "D"):
        if (warmest >= 22.0) and (months_ge_10 >= 4):
            third = "a"
        elif (months_ge_10 >= 4) and (warmest < 22.0):
            third = "b"
        elif months_ge_10 >= 1:
            third = "c"
        else:
            third = "d"
    else:
        third = ""

    details.update(group=main, second=second, third=third)
    return main + second + third, details


In [5]:
cityName = "Cairo"
year = "2020"
city=geocode_city(cityName)
print(city)
t = get_data(temp["tg"], city, "mean")
t = t[t["count_missing"] <= 10]
r = get_data(rain["rr"], city, "sum")
r = r[r["count_missing"] <= 10]
df = (
    t[["year", "month", "value"]]
    .rename(columns={"value": "temp"})
    .merge(
        r[["year", "month", "value"]].rename(columns={"value": "rain"}),
        on=["year", "month"],
        how="inner"
    )
)
classify_koppen_from_tables(temp_df=df, precip_df=df, latitude=city.latitude)

Location(name='القاهرة, مصر', latitude=30.0443879, longitude=31.2357257)


('BSh',
 {'annual_mean_temp': 23.82653522491455,
  'annual_precip': 37.709643602371216,
  'warmest': 30.76995086669922,
  'coldest': 16.011150360107422,
  'months_ge_10': 12,
  'precip_summer': 3.9700000286102295,
  'precip_winter': 33.739643573760986,
  'summer_share': 0.10527811056693712,
  'winter_share': 0.8947218894330629,
  'dryness_threshold_R': 47.6530704498291,
  'group': 'B',
  'sub1': 'S',
  'sub2': 'h'})

In [12]:
for cityName in ["Weymouth", "Swinoujscie", "Krakow", "Warszawa", "Genoa", "Bratislava", "Lisbon", "Cambridge", "Porto", "Nice"]:
    city=geocode_city(cityName)
    t = get_data(temp["tg"], city, "mean")
    t = t[t["count_missing"] <= 10]
    r = get_data(rain["rr"], city, "sum")
    r = r[r["count_missing"] <= 10]
    df = (
        t[["year", "month", "value"]]
        .rename(columns={"value": "temp"})
        .merge(
            r[["year", "month", "value"]].rename(columns={"value": "rain"}),
            on=["year", "month"],
            how="inner"
        )
    )
    print(city,classify_koppen_from_tables(temp_df=df, precip_df=df, latitude=city.latitude)[0])

Location(name='Weymouth, Dorset, England, United Kingdom', latitude=50.6096257, longitude=-2.4543424) Cfb
Location(name='Świnoujście, województwo zachodniopomorskie, Polska', latitude=53.8270328, longitude=14.3356892) Cfb
Location(name='Kraków, województwo małopolskie, Polska', latitude=50.0619474, longitude=19.9368564) Cfb
Location(name='Warszawa, województwo mazowieckie, Polska', latitude=52.2333742, longitude=21.0711489) Cfb
Location(name='Genova, Liguria, Italia', latitude=44.40726, longitude=8.9338624) Csa
Location(name='Bratislava, Bratislavský kraj, Slovensko', latitude=48.1516988, longitude=17.1093063) Cfb
Location(name='Lisboa, Portugal', latitude=38.7077507, longitude=-9.1365919) Csa
Location(name='Cambridge, Cambridgeshire, Cambridgeshire and Peterborough, England, United Kingdom', latitude=52.2055314, longitude=0.1186637) Cfb
Location(name='Porto, Portugal', latitude=41.1502195, longitude=-8.6103497) Csb
Location(name="Nice, Alpes-Maritimes, Provence-Alpes-Côte d'Azur, Fran

In [13]:
for cityName in  ["Bratislava", "Lisbon", "Cambridge", "Porto", "Antibes"]:
    city=geocode_city(cityName)
    t = get_data(temp["tg"], city, "mean")
    t = t[t["count_missing"] <= 10]
    r = get_data(rain["rr"], city, "sum")
    r = r[r["count_missing"] <= 10]
    print(t)
    df = (
        t[["year", "month", "value"]]
        .rename(columns={"value": "temp"})
        .merge(
            r[["year", "month", "value"]].rename(columns={"value": "rain"}),
            on=["year", "month"],
            how="inner"
        )
    )
    print(city,classify_koppen_from_tables(temp_df=df, precip_df=df, latitude=city.latitude)[0])

     year  month      value  count_nonmissing  count_missing
0    1950      1  -4.445161                31              0
1    1950      2   1.429643                28              0
2    1950      3   6.711613                31              0
3    1950      4  10.592334                30              0
4    1950      5  16.584192                31              0
..    ...    ...        ...               ...            ...
895  2024      8  24.936131                31              0
896  2024      9  18.021332                30              0
897  2024     10  12.399034                31              0
898  2024     11   4.711000                30              0
899  2024     12   2.564194                31              0

[900 rows x 5 columns]
Location(name='Bratislava, Bratislavský kraj, Slovensko', latitude=48.1516988, longitude=17.1093063) Cfb


GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Lisbon&format=json&limit=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=10)"))